# User-to-Item Recommendation System with TF-IDF and Lemmatization
We also built a user-to-item recsys with TF-IDF where we wanted to check whether reducing word to their base form would improve the representation of the wine content and better recommendations.

In [42]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [43]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sofiiaavetisian/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/sofiiaavetisian/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [44]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [45]:
wines_features = pd.read_csv(
    Path("../data/processed/wines_features.csv"),
    usecols=["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity", "content_text"]
)

ratings = pd.read_csv(
    Path("../dataset/XWines_Full_21M_ratings.csv"),
    usecols=["UserID", "WineID", "Rating"],
    dtype={"UserID": "int32", "WineID": "int32", "Rating": "float32"}
)

wines_features = wines_features.drop_duplicates(subset="WineID").copy()
wines_features = wines_features[wines_features["content_text"].notna()].copy()
wines_features = wines_features[wines_features["content_text"].str.strip() != ""].copy()

wines_features["WineID"] = wines_features["WineID"].astype("int32")
ratings["WineID"] = ratings["WineID"].astype("int32")

ratings = ratings.drop_duplicates(subset=["UserID", "WineID"]).copy()

print("wines_features:", wines_features.shape)
print("ratings:", ratings.shape)

wines_features: (100646, 8)
ratings: (20590800, 3)


In [46]:
from pathlib import Path

RANDOM_STATE = 42
TEST_FRACTION = 0.2

PROJECT_ROOT = Path.cwd().resolve()
for c in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (c / 'data' / 'results' / 'shared_split').exists():
        PROJECT_ROOT = c
        break

SPLIT_DIR = PROJECT_ROOT / 'data' / 'results' / 'shared_split'
ARMS_DIR = PROJECT_ROOT / 'bandits' / 'saved_arms'
ARMS_DIR.mkdir(parents=True, exist_ok=True)

train_pos = pd.read_csv(
    SPLIT_DIR / 'train_pos.csv',
    dtype={'UserID': 'int32', 'WineID': 'int32', 'Rating': 'float32'}
)
test_pos = pd.read_csv(
    SPLIT_DIR / 'test_pos.csv',
    dtype={'UserID': 'int32', 'WineID': 'int32', 'Rating': 'float32'}
)

shared_users = sorted(test_pos['UserID'].unique())

print("Loaded shared split from:", SPLIT_DIR)
print("train_pos:", train_pos.shape)
print("test_pos:", test_pos.shape)
print("shared users:", len(shared_users))

Loaded shared split from: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system/data/results/shared_split
train_pos: (56673, 3)
test_pos: (16646, 3)
shared users: 5000


In [47]:
def lemma_piece(piece):
    piece = piece.strip().lower()
    if not piece:
        return piece
    
    # lightweight multi-pass lemmatization
    piece = lemmatizer.lemmatize(piece, pos="n")
    piece = lemmatizer.lemmatize(piece, pos="v")
    piece = lemmatizer.lemmatize(piece, pos="a")
    return piece


def lemmatize_structured_token(token):
    parts = token.split("_")
    parts = [lemma_piece(part) for part in parts if part]
    return "_".join(parts)


def lemmatize_document(doc):
    tokens = str(doc).split()
    tokens = [lemmatize_structured_token(token) for token in tokens]
    return " ".join(tokens)

We transformed the wine description into lemmatized text where inflected word forms were reduced to their base form. Our goal was to reduce vocabulary sparsity and make similar descriptions easier to match in the TF-IDF space.

In [48]:
wines_features["content_text_lemma"] = wines_features["content_text"].apply(lemmatize_document)

In [49]:
wines_features[["content_text", "content_text_lemma"]].head(5)

,content_text,content_text_lemma
0,type_sparkling elaborate_varietal_100 body_med...,type_sparkle elaborate_varietal_100 body_mediu...
1,type_red elaborate_varietal_100 body_medium_bo...,type_red elaborate_varietal_100 body_medium_bo...
2,type_red elaborate_varietal_100 body_full_bodi...,type_red elaborate_varietal_100 body_full_body...
3,type_white elaborate_varietal_100 body_medium_...,type_white elaborate_varietal_100 body_medium_...
4,type_red elaborate_assemblage_bordeaux_red_ble...,type_red elaborate_assemblage_bordeaux_red_ble...


In [50]:
tfidf_lemma = TfidfVectorizer()
tfidf_lemma_matrix = tfidf_lemma.fit_transform(wines_features["content_text_lemma"])

In [51]:
tfidf_lemma = TfidfVectorizer(
    min_df=5,
    max_df=0.8,
    max_features=30000
)
tfidf_lemma_matrix = tfidf_lemma.fit_transform(wines_features["content_text_lemma"])

In [52]:
print(tfidf_lemma_matrix.shape)

(100646, 8725)


In [53]:
wines_features = wines_features.reset_index(drop=True)

wineid_to_idx = pd.Series(wines_features.index, index=wines_features["WineID"]).drop_duplicates()
idx_to_wineid = pd.Series(wines_features["WineID"].values, index=wines_features.index)

train_pos = train_pos[train_pos["WineID"].isin(wineid_to_idx.index)].copy()
test_pos = test_pos[test_pos["WineID"].isin(wineid_to_idx.index)].copy()

print("train_pos after filtering:", train_pos.shape)
print("test_pos after filtering:", test_pos.shape)

train_pos after filtering: (56673, 3)
test_pos after filtering: (16646, 3)


In [54]:
def build_user_profile_lemma(user_history):
    user_history = user_history[user_history["WineID"].isin(wineid_to_idx.index)].copy()

    item_indices = [wineid_to_idx[wine_id] for wine_id in user_history["WineID"]]
    weights = (user_history["Rating"] - 3.0).clip(lower=1.0).to_numpy()

    item_matrix = tfidf_lemma_matrix[item_indices]
    weighted_sum = item_matrix.multiply(weights.reshape(-1, 1)).sum(axis=0)
    profile = weighted_sum / weights.sum()

    return csr_matrix(profile)

In [55]:
user_profiles_lemma = {}
user_groups = list(train_pos.groupby("UserID"))

for i, (user_id, user_history) in enumerate(user_groups, start=1):
    user_profiles_lemma[user_id] = build_user_profile_lemma(user_history)
    if i % 500 == 0:
        print(f"Built {i}/{len(user_groups)} lemmatized TF-IDF user profiles")

print("built profiles:", len(user_profiles_lemma))

Built 500/5000 lemmatized TF-IDF user profiles
Built 1000/5000 lemmatized TF-IDF user profiles
Built 1500/5000 lemmatized TF-IDF user profiles
Built 2000/5000 lemmatized TF-IDF user profiles
Built 2500/5000 lemmatized TF-IDF user profiles
Built 3000/5000 lemmatized TF-IDF user profiles
Built 3500/5000 lemmatized TF-IDF user profiles
Built 4000/5000 lemmatized TF-IDF user profiles
Built 4500/5000 lemmatized TF-IDF user profiles
Built 5000/5000 lemmatized TF-IDF user profiles
built profiles: 5000


In [56]:
user_profiles_lemma = {}
user_groups = list(train_pos.groupby("UserID"))

for i, (user_id, user_history) in enumerate(user_groups, start=1):
    user_profiles_lemma[user_id] = build_user_profile_lemma(user_history)
    if i % 500 == 0:
        print(f"Built {i}/{len(user_groups)} lemmatized TF-IDF user profiles")

print("built profiles:", len(user_profiles_lemma))

Built 500/5000 lemmatized TF-IDF user profiles
Built 1000/5000 lemmatized TF-IDF user profiles
Built 1500/5000 lemmatized TF-IDF user profiles
Built 2000/5000 lemmatized TF-IDF user profiles
Built 2500/5000 lemmatized TF-IDF user profiles
Built 3000/5000 lemmatized TF-IDF user profiles
Built 3500/5000 lemmatized TF-IDF user profiles
Built 4000/5000 lemmatized TF-IDF user profiles
Built 4500/5000 lemmatized TF-IDF user profiles
Built 5000/5000 lemmatized TF-IDF user profiles
built profiles: 5000


In [57]:
train_seen = train_pos.groupby("UserID")["WineID"].apply(set).to_dict()
test_relevant = test_pos.groupby("UserID")["WineID"].apply(set).to_dict()

eval_users = sorted(set(user_profiles_lemma.keys()) & set(test_relevant.keys()))
print("evaluation users:", len(eval_users))

train_seen_idx = {
    user_id: [wineid_to_idx[w] for w in seen if w in wineid_to_idx]
    for user_id, seen in train_seen.items()
}

evaluation users: 5000


In [58]:
def recommend_user_lemma_ids(user_id, top_k=20, candidate_pool=400):
    if user_id not in user_profiles_lemma:
        return []

    profile = user_profiles_lemma[user_id]
    scores = linear_kernel(profile, tfidf_lemma_matrix).ravel()

    seen_idx = train_seen_idx.get(user_id, [])
    if seen_idx:
        scores[seen_idx] = -np.inf

    pool = min(candidate_pool, scores.size)
    candidate_idx = np.argpartition(scores, -pool)[-pool:]
    candidate_idx = candidate_idx[np.argsort(scores[candidate_idx])[::-1]]

    return [int(idx_to_wineid[i]) for i in candidate_idx[:top_k]]

In [59]:
def show_user_recommendations_lemma(user_id, top_k=10):
    rec_ids = recommend_user_lemma_ids(user_id, top_k=top_k)

    recs = wines_features[wines_features["WineID"].isin(rec_ids)][
        ["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity"]
    ].copy()

    recs["rank"] = recs["WineID"].map({wine_id: i + 1 for i, wine_id in enumerate(rec_ids)})
    recs = recs.sort_values("rank").reset_index(drop=True)

    return recs

In [60]:
sample_user = eval_users[0]
show_user_recommendations_lemma(sample_user, top_k=10)

,WineID,WineName,Type,Country,RegionName,Body,Acidity,rank
0,140018,Nebbiolo Terrazze Retiche di Sondrio,Red,Italy,Terrazze Retiche di Sondrio,Very full-bodied,High,1
1,138509,Boca,Red,Italy,Boca,Full-bodied,High,2
2,141894,Perno Barolo Riserva,Red,Italy,Barolo,Very full-bodied,High,3
3,145064,Bricco Parussi Barolo Riserva,Red,Italy,Barolo,Very full-bodied,High,4
4,150022,Bricco Ambrogio Barolo,Red,Italy,Barolo,Very full-bodied,High,5
5,139373,Barolo Bussia Munie Riserva,Red,Italy,Barolo,Very full-bodied,High,6
6,138435,Bricco delle Viole Barolo,Red,Italy,Barolo,Very full-bodied,High,7
7,139362,Ginestra Barolo,Red,Italy,Barolo,Very full-bodied,High,8
8,151385,Barolo Bussia Riserva,Red,Italy,Barolo,Very full-bodied,High,9
9,143101,Barolo Riserva del Fondatore,Red,Italy,Barolo,Very full-bodied,High,10


The lemmatized TF-IDF model returns a very focused recommendation list with all the wines being red French wines and coming from similar regions. This suggests that the model captures a clear regional and stylistic preference, although the recommendations may be somewhat narrow in diversity.

In [61]:
def precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / k

def recall_at_k(relevant, recommended, k):
    if len(relevant) == 0:
        return 0.0
    recommended_k = recommended[:k]
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / len(relevant)

def dcg_at_k(relevant, recommended, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            score += 1.0 / np.log2(i + 1)
    return score

def ndcg_at_k(relevant, recommended, k):
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg_at_k(relevant, recommended, k) / ideal_dcg

def average_precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    hits = 0
    score = 0.0

    for i, item in enumerate(recommended_k, start=1):
        if item in relevant:
            hits += 1
            score += hits / i

    denom = min(len(relevant), k)
    return score / denom if denom > 0 else 0.0

def hit_rate_at_k(relevant, recommended, k):
    return float(any(item in relevant for item in recommended[:k]))

def reciprocal_rank_at_k(relevant, recommended, k):
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            return 1.0 / i
    return 0.0

In [62]:
def evaluate_recommender(recommend_func, users, test_relevant_dict, ks=(5, 10, 20)):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_dict.get(user_id, set())
        if not relevant:
            continue

        recommended = recommend_func(user_id, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [63]:
lemma_eval = evaluate_recommender(
    recommend_func=recommend_user_lemma_ids,
    users=eval_users,
    test_relevant_dict=test_relevant,
    ks=(5, 10, 20)
)

lemma_summary = (
    lemma_eval
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="Lemma TF-IDF")
)

lemma_summary

Evaluated 500/5000 users
Evaluated 1000/5000 users
Evaluated 1500/5000 users
Evaluated 2000/5000 users
Evaluated 2500/5000 users
Evaluated 3000/5000 users
Evaluated 3500/5000 users
Evaluated 4000/5000 users
Evaluated 4500/5000 users
Evaluated 5000/5000 users


,Lemma TF-IDF
Precision@5,0.0020
Recall@5,0.0048
NDCG@5,0.0038
MAP@5,0.0026
HitRate@5,0.0100
MRR@5,0.0052
Precision@10,0.0018
Recall@10,0.0080
NDCG@10,0.0051
MAP@10,0.0030


In [64]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
lemma_summary.to_csv("../data/results/lemma_summary.csv")

In [65]:
wines_features["eval_signature"] = (
    wines_features["Type"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Country"].astype(str).str.lower().str.strip() + "|" +
    wines_features["RegionName"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Body"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Acidity"].astype(str).str.lower().str.strip()
)

wineid_to_signature = dict(zip(wines_features["WineID"], wines_features["eval_signature"]))

test_relevant_signatures = {
    user_id: {wineid_to_signature[w] for w in wine_ids if w in wineid_to_signature}
    for user_id, wine_ids in test_relevant.items()
}

In [66]:
def recommended_ids_to_signatures(recommended_ids, top_k=None):
    signatures = []
    seen_signatures = set()

    for wine_id in recommended_ids:
        if wine_id not in wineid_to_signature:
            continue

        signature = wineid_to_signature[wine_id]
        if signature in seen_signatures:
            continue

        signatures.append(signature)
        seen_signatures.add(signature)

        if top_k is not None and len(signatures) == top_k:
            break

    return signatures

In [67]:
def evaluate_recommender_soft(recommend_func, users, test_relevant_signatures_dict, ks=(5, 10, 20), oversample_factor=5):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_signatures_dict.get(user_id, set())
        if not relevant:
            continue

        recommended_ids = recommend_func(user_id, top_k=max_k * oversample_factor)
        recommended = recommended_ids_to_signatures(recommended_ids, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Soft-evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [68]:
lemma_eval_soft = evaluate_recommender_soft(
    recommend_func=recommend_user_lemma_ids,
    users=eval_users,
    test_relevant_signatures_dict=test_relevant_signatures,
    ks=(5, 10, 20),
    oversample_factor=5
)

lemma_summary_soft = (
    lemma_eval_soft
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="Lemma TF-IDF Soft")
)

lemma_summary_soft

Soft-evaluated 500/5000 users
Soft-evaluated 1000/5000 users
Soft-evaluated 1500/5000 users
Soft-evaluated 2000/5000 users
Soft-evaluated 2500/5000 users
Soft-evaluated 3000/5000 users
Soft-evaluated 3500/5000 users
Soft-evaluated 4000/5000 users
Soft-evaluated 4500/5000 users
Soft-evaluated 5000/5000 users


,Lemma TF-IDF Soft
Precision@5,0.0539
Recall@5,0.1050
NDCG@5,0.1066
MAP@5,0.0772
HitRate@5,0.2428
MRR@5,0.1711
Precision@10,0.0331
Recall@10,0.1271
NDCG@10,0.1132
MAP@10,0.0784


In [69]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
lemma_eval_soft.to_csv("../data/results/lemma_eval_soft.csv", index=False)
lemma_summary_soft.to_csv("../data/results/lemma_summary_soft.csv")

# Save the models

In [70]:
from pathlib import Path
import pandas as pd

ARMS_DIR = Path("../bandits/saved_arms")
ARMS_DIR.mkdir(parents=True, exist_ok=True)

def export_arm_recs(recommend_fn, users, out_csv, top_k=100):
    rows = []
    for uid in users:
        recs = recommend_fn(int(uid), top_k=top_k)
        for r, wid in enumerate(recs, start=1):
            rows.append({"UserID": int(uid), "rank": int(r), "WineID": int(wid)})
    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_csv, index=False)
    print(f"Saved {len(out_df):,} rows -> {out_csv}")


In [71]:
export_arm_recs(
    recommend_fn=recommend_user_lemma_ids,
    users=shared_users,
    out_csv=ARMS_DIR / "content_lemma_tfidf_recs.csv",
    top_k=100
)


Saved 500,000 rows -> ../bandits/saved_arms/content_lemma_tfidf_recs.csv
